In [132]:
# Here we import the packages we will need
import datetime
import pandas as pd
import argparse
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import PolynomialFeatures, LabelEncoder, OneHotEncoder
from sklearn import metrics, linear_model
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
import mlflow
import mlflow.sklearn
import logging
import os
import pickle
import json
import time

In [133]:
# set up the argument parser
parser = argparse.ArgumentParser(description='Parse the parameters for the polynomial regression')
parser.add_argument('num_alphas', metavar='N', type=int, help='Number of Lasso penalty increments')
order = 1
#args = parser.parse_args()
#num_alpha_increments = args[0]
# Uncomment the two lines above and comment the line below to run this script from the command prompt or as part of an 
# MLFlow pipeline
num_alphas = 20

In [134]:
# configure logger
logname = "polynomial_regression.txt"
logging.basicConfig(filename=logname,
                    filemode='w',
                    format='%(asctime)s %(levelname)s %(message)s',
                    datefmt='%H:%M:%S',
                    level=logging.DEBUG)
logging.getLogger('matplotlib.font_manager').disabled = True
logging.info("Flight Departure Delays Polynomial Regression Model Log")

In [135]:
# read the data file
df = pd.read_csv("C:/Users/uyen/Desktop/d602/notebook/T_ONTIME_REPORTING.csv")
tab_info=pd.DataFrame(df.dtypes).T.rename(index={0:'column type'})

In [136]:
df.columns = ["YEAR", "MONTH", "DAY", "DAY_OF_WEEK", "ORG_AIRPORT", "DEST_AIRPORT", "SCHEDULED_DEPARTURE", "DEPARTURE_TIME",
"DEPARTURE_DELAY", "SCHEDULED_ARRIVAL", "ARRIVAL_TIME", "ARRIVAL_DELAY"]

In [137]:
def grab_month_year(df:pd.DataFrame) -> tuple:
    """
    grab_month_year is a function to extract the month and year of the flights in the departure delay dataset.

    Parameters
    ----------
    df : pd.DataFrame
        the input data set in Pandas data frame format.

    Returns
    -------
    tuple
        (month,year) of the data set.

    Raises
    ------
    Exception
        If more than one month or year are found in the data set.
    """
    months = pd.unique(df['MONTH'])
    years = pd.unique(df['YEAR'])
    if len(months) >1:
        raise Exception("Multiple months found in data set, only one acceptable")
    else:
        month = int(months[0])
    if len(years) > 1:
        raise Exception("Multiple years found in data set, only one acceptable")
    else:
        year = int(years[0])
    return (month, year)

In [138]:
def format_hour(string: str) -> datetime:
    """
    format_hour is a function to convert an 'HHMM' string input to a time in datetime format.

    Parameters
    ----------
    string : string
        An hour and minute in 'HHMM' format.

    Returns
    -------
    datetime
        An hour and minute (datetime.time).  Returns nan if input string is null.

    """    
    if pd.isnull(string):
        return np.nan
    else:
        if string == 2400: string = 0
        string = "{0:04d}".format(int(string))
        hour = datetime.time(int(string[0:2]), int(string[2:4]))
        return hour

In [139]:
def combine_date_hour(x: list) -> datetime:
    """
    combine_date_hour is a function that combines a date and time to produce a datetime.datetime

    Parameters
    ----------
    x : list
        A list containing a date and a time in datetime format.

    Returns
    -------
    datetime
        A combined date and time in datetime format. Returns nan if time is null.

    """
    if pd.isnull(x.iloc[0]) or pd.isnull(x.iloc[1]):
        return np.nan
    else:
        return datetime.datetime.combine(x.iloc[0],x.iloc[1])

In [140]:
def create_flight_time(df: pd.DataFrame, col: str) -> pd.Series:
    """
    create_flight_time is a function that combines two columns of a data frame to produce a datetime.datetime series.

    Parameters
    ----------
    df : pd.DataFrame
        A data frame containing flight departure delay data
    col: string
        The name of one of the columns in the data frame containing flight departure delay data

    Returns
    -------
    pd.Series
        A Pandas series of datetimes with combined date and time

    """
    list = []
    for index, cols in df[['DATE', col]].iterrows():
        if pd.isnull(cols.iloc[1]):
            list.append(np.nan)
        elif float(cols.iloc[1]) == 2400:
            cols.iloc[0] += datetime.timedelta(days=1)
            cols.iloc[1] = datetime.time(0,0)
            list.append(combine_date_hour(cols))
        else:
            cols.iloc[1] = format_hour(cols.iloc[1])
            list.append(combine_date_hour(cols))
    return pd.Series(list)

In [141]:
def create_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    create_df is a function that wrangles data within a flight departure delay data frame into the format needed for ML training.

    Parameters
    ----------
    df : pd.DataFrame
        A data frame containing flight departure delay data

    Returns
    -------
    pd.DataFrame
        A Pandas data frame with modified columns and data formats suitable for regression model training

    """
    df2 = df[['SCHEDULED_DEPARTURE','SCHEDULED_ARRIVAL',
                                    'DEST_AIRPORT','DEPARTURE_DELAY']]
    df2 = df2.dropna(how = 'any')
    df2.loc[:,'weekday'] = df2['SCHEDULED_DEPARTURE'].apply(lambda x:x.weekday())
    #____________________
    # delete delays > 1h
    df2.loc[:,'DEPARTURE_DELAY'] = df2['DEPARTURE_DELAY'].apply(lambda x:x if x < 60 else np.nan)
    df2 = df2.dropna(how = 'any')
    #_________________
    # formating times
    fct = lambda x:x.hour*3600+x.minute*60+x.second
    df2.loc[:,'hour_depart'] = df2['SCHEDULED_DEPARTURE'].apply(lambda x:x.time())
    df2.loc[:,'hour_depart'] = df2['hour_depart'].apply(fct)
    df2.loc[:,'hour_arrive'] = df2['SCHEDULED_ARRIVAL'].apply(fct)
    df2 = df2[['hour_depart','hour_arrive',
            'DEST_AIRPORT','DEPARTURE_DELAY','weekday']]
    df3 = df2.groupby(['hour_depart', 'hour_arrive', 'DEST_AIRPORT'],
                      as_index = False).mean()
    return df3

In [142]:
nowdate = datetime.date.today()
# creates an experiment name that changes every day
experiment_name = "Airport Departure Delays, experiment run on " + str(nowdate)
# creates new experiment if there is not one yet today, otherwise sets the experiment to the existing one for today
experiment = mlflow.set_experiment(experiment_name)
run_name = "Run started at " + datetime.datetime.now().strftime("%H:%M")

In [143]:
df['DATE'] = pd.to_datetime(df[['YEAR','MONTH', 'DAY']])

In [144]:
df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,ORG_AIRPORT,DEST_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DATE
0,2025,6,1,7,ABQ,BUR,1725,1739.0,14.0,1830,1835.0,5.0,2025-06-01
1,2025,6,1,7,ABQ,LAX,600,554.0,-6.0,713,649.0,-24.0,2025-06-01
2,2025,6,1,7,ABQ,LAX,650,650.0,0.0,800,750.0,-10.0,2025-06-01
3,2025,6,1,7,ABQ,LAX,1154,1244.0,50.0,1310,1358.0,48.0,2025-06-01
4,2025,6,1,7,ABQ,LAX,1314,1315.0,1.0,1427,1451.0,24.0,2025-06-01


In [145]:
(month,year) = grab_month_year(df)

In [146]:
month

6

In [147]:
year

2025

In [148]:
logging.info("Month and year of data: %s %s", month, year)

In [149]:
df['SCHEDULED_DEPARTURE'] = create_flight_time(df, 'SCHEDULED_DEPARTURE')

In [150]:
df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,ORG_AIRPORT,DEST_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DATE
0,2025,6,1,7,ABQ,BUR,2025-06-01 17:25:00,1739.0,14.0,1830,1835.0,5.0,2025-06-01
1,2025,6,1,7,ABQ,LAX,2025-06-01 06:00:00,554.0,-6.0,713,649.0,-24.0,2025-06-01
2,2025,6,1,7,ABQ,LAX,2025-06-01 06:50:00,650.0,0.0,800,750.0,-10.0,2025-06-01
3,2025,6,1,7,ABQ,LAX,2025-06-01 11:54:00,1244.0,50.0,1310,1358.0,48.0,2025-06-01
4,2025,6,1,7,ABQ,LAX,2025-06-01 13:14:00,1315.0,1.0,1427,1451.0,24.0,2025-06-01


In [151]:
df['DEPARTURE_TIME'] = df['DEPARTURE_TIME'].apply(format_hour)

In [152]:
df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,ORG_AIRPORT,DEST_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DATE
0,2025,6,1,7,ABQ,BUR,2025-06-01 17:25:00,17:39:00,14.0,1830,1835.0,5.0,2025-06-01
1,2025,6,1,7,ABQ,LAX,2025-06-01 06:00:00,05:54:00,-6.0,713,649.0,-24.0,2025-06-01
2,2025,6,1,7,ABQ,LAX,2025-06-01 06:50:00,06:50:00,0.0,800,750.0,-10.0,2025-06-01
3,2025,6,1,7,ABQ,LAX,2025-06-01 11:54:00,12:44:00,50.0,1310,1358.0,48.0,2025-06-01
4,2025,6,1,7,ABQ,LAX,2025-06-01 13:14:00,13:15:00,1.0,1427,1451.0,24.0,2025-06-01


In [153]:
df['SCHEDULED_ARRIVAL'] = df['SCHEDULED_ARRIVAL'].apply(format_hour)

In [154]:
df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,ORG_AIRPORT,DEST_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DATE
0,2025,6,1,7,ABQ,BUR,2025-06-01 17:25:00,17:39:00,14.0,18:30:00,1835.0,5.0,2025-06-01
1,2025,6,1,7,ABQ,LAX,2025-06-01 06:00:00,05:54:00,-6.0,07:13:00,649.0,-24.0,2025-06-01
2,2025,6,1,7,ABQ,LAX,2025-06-01 06:50:00,06:50:00,0.0,08:00:00,750.0,-10.0,2025-06-01
3,2025,6,1,7,ABQ,LAX,2025-06-01 11:54:00,12:44:00,50.0,13:10:00,1358.0,48.0,2025-06-01
4,2025,6,1,7,ABQ,LAX,2025-06-01 13:14:00,13:15:00,1.0,14:27:00,1451.0,24.0,2025-06-01


In [155]:
df['ARRIVAL_TIME'] = df['ARRIVAL_TIME'].apply(format_hour)

In [156]:
df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,ORG_AIRPORT,DEST_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DATE
0,2025,6,1,7,ABQ,BUR,2025-06-01 17:25:00,17:39:00,14.0,18:30:00,18:35:00,5.0,2025-06-01
1,2025,6,1,7,ABQ,LAX,2025-06-01 06:00:00,05:54:00,-6.0,07:13:00,06:49:00,-24.0,2025-06-01
2,2025,6,1,7,ABQ,LAX,2025-06-01 06:50:00,06:50:00,0.0,08:00:00,07:50:00,-10.0,2025-06-01
3,2025,6,1,7,ABQ,LAX,2025-06-01 11:54:00,12:44:00,50.0,13:10:00,13:58:00,48.0,2025-06-01
4,2025,6,1,7,ABQ,LAX,2025-06-01 13:14:00,13:15:00,1.0,14:27:00,14:51:00,24.0,2025-06-01


In [157]:
df = df[df["ORG_AIRPORT"] == "LAX"]

In [158]:
# define training data as the first 3 weeks of the month, and test data as that from the fourth week of the month
df_train = df[df['SCHEDULED_DEPARTURE'].apply(lambda x:x.date()) < datetime.date(year, month, 23)]
df_test  = df[df['SCHEDULED_DEPARTURE'].apply(lambda x:x.date()) > datetime.date(year, month, 23)]

In [159]:
df_train

,YEAR,MONTH,DAY,DAY_OF_WEEK,ORG_AIRPORT,DEST_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DATE
1061,2025,6,1,7,LAX,ABQ,2025-06-01 08:45:00,08:31:00,-14.0,11:50:00,12:08:00,18.0,2025-06-01
1062,2025,6,1,7,LAX,ABQ,2025-06-01 09:00:00,08:59:00,-1.0,12:00:00,12:17:00,17.0,2025-06-01
1063,2025,6,1,7,LAX,ABQ,2025-06-01 09:44:00,09:36:00,-8.0,12:44:00,12:50:00,6.0,2025-06-01
1064,2025,6,1,7,LAX,ABQ,2025-06-01 14:40:00,15:06:00,26.0,17:33:00,18:04:00,31.0,2025-06-01
1065,2025,6,1,7,LAX,ABQ,2025-06-01 14:59:00,15:37:00,38.0,17:58:00,18:29:00,31.0,2025-06-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
80592,2025,6,22,7,LAX,TUS,2025-06-22 14:10:00,14:30:00,20.0,15:44:00,15:53:00,9.0,2025-06-22
80593,2025,6,22,7,LAX,TUS,2025-06-22 14:58:00,14:50:00,-8.0,16:27:00,16:13:00,-14.0,2025-06-22
80594,2025,6,22,7,LAX,TUS,2025-06-22 18:18:00,18:08:00,-10.0,19:49:00,19:30:00,-19.0,2025-06-22
80595,2025,6,22,7,LAX,TUS,2025-06-22 19:00:00,22:27:00,207.0,20:33:00,23:50:00,197.0,2025-06-22


In [165]:
df_train[(df_train["DEPARTURE_DELAY"] == -7.0) & (df_train["DEST_AIRPORT"] == 'DTW')]

,YEAR,MONTH,DAY,DAY_OF_WEEK,ORG_AIRPORT,DEST_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DATE
5039,2025,6,2,1,LAX,DTW,2025-06-02 05:35:00,05:28:00,-7.0,13:01:00,12:46:00,-15.0,2025-06-02
5042,2025,6,2,1,LAX,DTW,2025-06-02 11:47:00,11:40:00,-7.0,19:11:00,19:07:00,-4.0,2025-06-02
12349,2025,6,4,3,LAX,DTW,2025-06-04 15:52:00,15:45:00,-7.0,23:18:00,23:35:00,17.0,2025-06-04
23769,2025,6,7,6,LAX,DTW,2025-06-07 22:35:00,22:28:00,-7.0,06:00:00,05:49:00,-11.0,2025-06-07
27269,2025,6,8,7,LAX,DTW,2025-06-08 15:36:00,15:29:00,-7.0,23:04:00,22:47:00,-17.0,2025-06-08
31159,2025,6,9,1,LAX,DTW,2025-06-09 09:59:00,09:52:00,-7.0,17:23:00,17:13:00,-10.0,2025-06-09
42407,2025,6,12,4,LAX,DTW,2025-06-12 12:09:00,12:02:00,-7.0,19:31:00,19:31:00,0.0,2025-06-12
50173,2025,6,14,6,LAX,DTW,2025-06-14 07:35:00,07:28:00,-7.0,14:56:00,14:53:00,-3.0,2025-06-14
57574,2025,6,16,1,LAX,DTW,2025-06-16 11:14:00,11:07:00,-7.0,18:37:00,18:36:00,-1.0,2025-06-16
57576,2025,6,16,1,LAX,DTW,2025-06-16 15:36:00,15:29:00,-7.0,23:04:00,22:29:00,-35.0,2025-06-16


In [160]:
df3 = create_df(df_train)

In [161]:
df3.head()

,hour_depart,hour_arrive,DEST_AIRPORT,DEPARTURE_DELAY,weekday
0,1200,28320,DTW,-7.000000,3.750000
1,1380,23700,ORD,22.000000,5.000000
2,1800,23940,ORD,-7.000000,3.000000
3,1800,24120,ORD,2.571429,2.428571
4,1920,23880,ORD,6.375000,3.500000


In [ ]:
import datetime

# Create a date object
date_obj = datetime.date(2025, 5, 8) # May 8, 2025 (a Thursday)

# Use the .weekday() method
day_number = date_obj.weekday()

print(f"The day number is: {day_number}")

The day number is: 3


In [167]:
date_obj

datetime.date(2025, 5, 8)